In [ ]:
### standard libraries
import time
import numpy as np
import matplotlib.pyplot as plt

### my modules
from fast_volume_align_ot import utils, plots, transforms, distances, signals

In [ ]:
def translate_3d_new(array, phid, p, thid):

    tmp = np.roll(array, (phid, 0, 0), axis=(0))
    res = np.zeros(array.shape, dtype=np.float64) #they are real, right?
    
    for k in range(res.shape[0]):
        for j in range(res.shape[2]):
            if j-thid >= 0:
                jj = j - thid
                kk = k
            else:
                jj = thid-j
                kk = (k-phid + res.shape[0]//2) % res.shape[0]
                
            res[k,:,j] = array[kk, :, jj]
            
    return res

In [ ]:
L = 34
sigma = 1.5

f = utils.generate_centered_gaussian(L=L, d=3, sigma=sigma)
f_1 = transforms.translate_3d(f, 0, 5, 0)     # move centered blob down
f_2 = transforms.translate_3d(f, 0, -7, -10)  # move blob far left and up
f_3 = transforms.translate_3d(f, 5, 0, 5)    # move blob back and right
F = f_1 + f_2 + f_3

F = signals.Signal(F, d=3).preprocess_signals()[0]
Pf = transforms.Transform(F, d=3).radon_transform()[0]

thetas = np.linspace(0, np.pi, L, endpoint=True)
phis = np.linspace(0, 2*np.pi, L, endpoint=False)

n_theta = len(thetas)
n_phi = len(phis)

s = 4
t = 5
phi = s*2*np.pi/len(phis)
theta = t*np.pi/len(thetas)

G = transforms.rotate_voxel_zyz(F, 0, theta, phi)
Pg = transforms.Transform(G, d=3).radon_transform()[0]


print('\t original')
plots.orthogonal_2D_projection_plot(F)
print('\t rotated')
plots.orthogonal_2D_projection_plot(G)

In [ ]:
# L = 68
# sigma = 2

# f = utils.generate_centered_gaussian(L=L, d=3, sigma=sigma)
# f_1 = transforms.translate_3d(f, 0, 10, 0)
# f_2 = transforms.translate_3d(f, 0, -15, -20)
# f_3 = transforms.translate_3d(f, 10, 0, 10)
# F = f_1 + f_2 + f_3

# F = signals.Signal(F, d=3).preprocess_signals()[0]

# thetas = np.linspace(0, np.pi, L, endpoint=True)
# phis = np.linspace(0, 2*np.pi, L, endpoint=False)

# omega = np.deg2rad(0)
# theta = np.deg2rad(45)
# phi = np.deg2rad(0)

# G = transforms.rotate_voxel_zyz(F, omega, theta, phi, order=1)
# G = signals.Signal(G, d=3).preprocess_signals()[0]

# print('\t original')
# plots.orthogonal_2D_projection_plot(F)
# print('\t rotated')
# plots.orthogonal_2D_projection_plot(G)


# # U = transforms.Transform(F, d=3, apply_ramp=False).inverse_cdf_transform()[0]
# # V = transforms.Transform(G, d=3, apply_ramp=False).inverse_cdf_transform()[0]

# # U = transforms.Transform(F, d=3, apply_ramp=False).radon_transform()[0]
# # V = transforms.Transform(G, d=3, apply_ramp=False).radon_transform()[0]

In [ ]:
### need to find vectorize or fast implementation of this
### does the interpolation need to be in the numpy or Euclidean? do I need to rotate with numpy ?
### can change len(thetas) to variable n_theta

t_start = time.time()


tau = np.zeros((n_phi, n_theta, n_phi, n_theta))
sigma = np.zeros((n_phi, n_theta, n_phi, n_theta))


for j in range(n_phi):
    p1 = phis[j]
    for k in range(n_theta):
        t1 = thetas[k]
        for s in range(n_phi):
            p2 = phis[s]
            for t in range(n_theta):
                t2 = thetas[t]

                v = np.array(
                    [
                        [np.sin(t1) * np.cos(p1)],
                        [np.sin(p1) * np.sin(t1)],
                        [np.cos(t1)]
                    ]
                )

                R1 = transforms.rotate_y(t2)
                R2 = transforms.rotate_z(p2)

                v_rot = R2 @ R1 @ v

                ### EV: arccos has a inf error somewhere. needs [-1, 1]?
                t3 = np.arccos(v_rot[2])
                p3 = np.atan2(v_rot[0], v_rot[1]) + np.pi

                t_new = np.argmin(abs(thetas - t3))
                p_new = np.argmin(abs(phis - p3))

                ### EV: also this causes a +/- inf error somewhere
                # t_new = np.floor( t3 * n_theta / (np.pi) )[0]  # EV: floor in case rounding index is out bounds
                # p_new = np.floor( p3 * n_phi / (2*np.pi) )[0]

                sigma[j, k, s, t] = t_new
                tau[j, k, s, t] = p_new


tau = tau.astype('int')
sigma = sigma.astype('int')


print(time.time() - t_start)

In [ ]:
# np.array([4.5]).astype('int'), np.round(4.5).astype('int')

In [ ]:
err = 0

for j in range(n_phi):
    for k in range(n_theta):
        err += np.linalg.norm( Pg[k,:,j] - Pf[tau[j,k,s,t], :, sigma[j,k,s,t]])

print(err / np.linalg.norm(Pg.flatten()))

In [ ]:
j,k,s,t

In [ ]:
# """
# for fiber in F:
#     fiber = F[phi, :, theta]:
    
#     for point in fiber:


# snap rotated point to closest point on grid to generate the correct rotated version (nearest neighbor)

# """